In [ ]:
import pickle
import numpy as np 
import pandas as pd
import datasets
from itertools import combinations, zip_longest, product
from collections import defaultdict
from scipy.stats import spearmanr, pearsonr
import seaborn as sns
from glob import glob
from os.path import basename, dirname
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import plotly.express as px
import copy

In [ ]:
# define a colormap that makes relevant correlations easier to see
# i.e. low to no correlation |s| < 0.5 => grey color

colors = [
    (0, "darkblue"),   # -1
    (0.25, "lightblue"),
    (0.35, "lightgray"),  # around -0.3 to 0.3 all grayish 
    (0.65, "lightgray"),
    (0.75, "salmon"),
    (1, "darkred")      # +1
]

cmap = LinearSegmentedColormap.from_list("custom_diverging", colors)


lang_map = {"eng": "en",
            "fra": "fr",
            "fin": "fi",
            "zho": "zh",
            "ita": "it",
            "deu": "de",
            "vie": "vi",
            "jpn": "ja",
            "ukr": "uk",
            "ell": "el",
            "hin": "hi",
            "bul": "bg",
            "fas": "fa"}
lang_map2 = {v:k for k,v in lang_map.items()}

In [ ]:
def load_data_and_matrix(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    with open(path.replace("results", "fitted"), "rb") as f:
        model = pickle.load(f)
    #print(model["V"].shape)
    try:  # ICA
        return data, model.components_
    except:  # SCA
        return data, model["V"]

In [ ]:
def plot_the_bar_graph(d, method, data_name, model_name):

    icas = np.array(d[f"{method}1-{method}2"])
    icas_shuf = np.array(d[f"{method}1-shuffled({method}2)"])
    
    # what to plot
    means_ica = np.mean(np.abs(icas), axis = 0)
    means_ica_shuf = np.mean(np.abs(icas_shuf), axis = 0)
    std_ica = np.std(np.abs(icas), axis = 0)
    std_ica_shuf = np.std(np.abs(icas_shuf), axis = 0)
    assert len(means_ica) > 1, "Took the mean along the wrong axis?"

    # two plots
    fig = px.bar(means_ica, error_y=std_ica)
    fig.update_layout(
        title=dict(
            text=f"{data_name}, {model_name}"
        ),
        xaxis=dict(
            title=dict(
                text="ISPCA dimension"
            )
        ),
        yaxis=dict(
            title=dict(
                text="Mean absolute distance on given dimension, paired data points"
            )
        ),
    )
    fig.show()
    """
    fig = px.bar(means_ica_shuf, error_y=std_ica_shuf)
    fig.update_layout(
        title=dict(
            text=f"{data_name}, {model_name}"
        ),
        xaxis=dict(
            title=dict(
                text="ISPCA dimension"
            )
        ),
        yaxis=dict(
            title=dict(
                text="Mean absolute distance on given dimension, shuffled data points."
            )
        ),
    )
    fig.show()
    """

def gini_coefficient_and_lorenz_curve(distribution):
    sorted_dist = np.sort(distribution)
    n = len(sorted_dist)
    cum_sum = np.cumsum(sorted_dist)
    y = cum_sum / np.sum(sorted_dist)
    x = np.arange(len(distribution)) / len(distribution)
    gini = (n-1)/n - (2 * np.sum((n - np.arange(1, n+1)) * sorted_dist)) / (n * np.sum(sorted_dist))
    return gini, (x,y)

def kurtosis_and_skew(distribution):
    return kurtosis(distribution, fisher=False), skew(distribution)

def coefficient_of_variation(distribution):
    return np.std(distribution) / np.mean(distribution)

def peak_to_mean_ratio(distribution):
    return np.max(distribution) / np.mean(distribution)

def peak_dimensions_detection(distribution, sigma=3, method="argmax"):
    assert len(distribution) > 1, f"Length of dist == {len(distribution)}, should be dim_ISPCA"
    if method == "z-score":
        arr = distribution
        mean = np.mean(arr)
        std = np.std(arr)
        threshold = mean+sigma*std
        z_scores = (arr - mean) / std
        outlier_indices = np.where(np.abs(z_scores) > threshold)[0]
        return outlier_indices
    if method == "iqr":
        q1 = np.percentile(distribution, 25)
        q3 = np.percentile(distribution, 75)
        iqr = q3 - q1
        outliers = np.where(distribution > q3 + sigma * iqr)[0] # [0] for 1-tuple
        return outliers
    if method == "argmax":
        max_idx = np.abs(distribution).argmax()
        return [max_idx]


def calculate_statistics(d, method, prints=False):
    # ICA(data1)-ICA(data2)
    icas = np.array(d[f"{method}1-{method}2"])
    # ICA(data1)- shuffled(ICA(data2))  => to differentiate semantics vs. wanted quality
    icas_shuf = np.array(d[f"{method}1-shuffled({method}2)"])
    # mean(abs(x)) from both
    icas_mean = np.mean(np.abs(icas), axis = 0)
    icas_shuf_mean = np.mean(np.abs(icas_shuf), axis = 0)
    assert len(icas_mean) > 1, "Took the mean along the wrong axis?"

    # find the peaky dimension(s)
    # depending on the application, you can also use the shuffled mean for this
    # but not 
    contributing_dimensions = peak_dimensions_detection(icas_mean)
    if prints:
        print(f"Peaky dimensions are {contributing_dimensions}")
    # calculate metrics
    gini, lorenz_curve = gini_coefficient_and_lorenz_curve(icas_mean)
    var = coefficient_of_variation(icas_mean)
    peak = peak_to_mean_ratio(icas_mean)
    if prints:
        print(f"Metrics are:\n\tGini:{gini}\n\tVar:{var}\n\tPeak:{peak}")
    return contributing_dimensions, {"gini": gini, "var": var, "peak": peak}

def check_consistency_of_statistics(list_of_results, threshold=0.01, prints=False):  # 0.025
    is_consistent = {}
    # chatgpt helped here
    # Collect values per key
    data = defaultdict(list)
    for d in list_of_results:
        for k, v in d.items():
            data[k].append(float(v))

    summary = {}
    for k, vals in data.items():
        mean = np.mean(vals)
        std = np.std(vals)
        cv = std / mean if mean != 0 else np.inf
        summary[k] = {'mean': mean, 'std': std, 'cv': cv}
        if threshold is not None and cv > threshold:
            if prints:
                print(f"⚠️  '{k}' varies a lot: CV={cv:.3f} (> {threshold})")
            is_consistent[k] = False
        else:
            if prints:
                print(f"✅ okay, variation in {k} is {cv}")
            is_consistent[k] = True
    if all([p for p in is_consistent.values()]) and prints is True:
        print(f"All metrics consistent ✅")
    return summary, is_consistent
    

def build_annot(values, pvals, threshold=0.05):
    print(pvals)
    annot = np.empty_like(values, dtype=object)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            star = "*" if pvals[i, j] < threshold else ""
            annot[i, j] = f"{values[i, j]:.2f}{star}"
    return annot

def plot_correlation_heatmap(result_matrix, peaky_dimensions, title="Correlation heatmap", p_values=None):
    dim_a, dim_b = peaky_dimensions
    if p_values is None:
        mask = np.tril(np.ones_like(result_matrix, dtype=bool))
        ax = sns.heatmap(result_matrix, cmap=cmap, vmin=-1, vmax=1,xticklabels=True, yticklabels=True, annot=True, fmt='.2f', mask=mask)
    else:
        annot = build_annot(result_matrix, p_values)
        mask = np.tril(np.ones_like(result_matrix, dtype=bool))
        ax = sns.heatmap(result_matrix, cmap=cmap, vmin=-1, vmax=1,xticklabels=True, yticklabels=True, annot=annot, fmt='', mask=mask)
    ax.set_title(str(title))
    for x1,y1 in zip_longest(dim_b, dim_a):  # this order for the axes
        if x1 is not None:
            highlight_x = x1
            xlabels = ax.get_xticklabels()
            xlabels[highlight_x].set_fontweight('bold')
            xlabels[highlight_x].set_color('red')
        if y1 is not None:
            ylabels = ax.get_yticklabels()
            highlight_y = y1
            ylabels[highlight_y].set_fontweight('bold')
            ylabels[highlight_y].set_color('red')
    ax.figure.canvas.draw()
    plt.show()
    plt.close()


def check_if_a_permutation_matrix(m):
    # i.e. just check if there are only one 1 in all rows and columns, is it a shuffled identity matrix
    if not set(np.unique(m)) == set([0,1]):
        # we have continuous values
        m = (m != 0).astype(int)
    if (np.sum(m, axis=0) == np.ones((1, m.shape[0]))).all() and (np.sum(m, axis=1) == np.ones((m.shape[1], 1))).all():
        #print("YES a permutation")
        return True
    # what the problem is?
    print("NOT a permutation, this is why")
    rows = np.sum(m, axis=0)
    columns = np.sum(m, axis=1)
    for a in [rows, columns]:
        # Find indices where value != 1
        mask = a != 1
        # Get indices and values
        indices = np.where(mask)[0]
        values = a[mask]
        index_value_pairs = {int(k):int(v) for k,v in zip(indices, values)}
        print(f"errors in: {index_value_pairs}")
    return False

def from_correlation_values_to_boolean(m, method="argmax", th=0.80, outlier_level=2):
    # from row-to-row correlation to 
    assert method in ["argmax", "threshold", "outlier", "best-k", "greedy"], "method badly defined in from_correlation_values_to_boolean()"
    # create a mask to fill with values
    mask = np.zeros_like(m, dtype=int)
    if method == "argmax":
        # For each row, find index of max absolute value
        max_idx = np.abs(m).argmax(axis=1)
        # Use advanced indexing to set the correct positions to 1
        mask[np.arange(m.shape[0]), max_idx] = 1
        return mask
    if method == "threshold":
        # set all indices to one where correlation is more than th
        for i, row in enumerate(m):
            mask[i,:] = [int(v > th) for v in row]
        return mask
    if method == "outlier":
        # same as above but calculate th row by row
        for i, row in enumerate(m):
            mean,std = np.mean(row), np.std(row)
            mask[i,:] = [int(v > mean+outlier_level*std) for v in row]
        return mask
    if method == "best-k":
        # set top values to 1
        k = m.shape[0]  # square matrix
        topk_idx_partition = np.argpartition(np.abs(m), -k, axis=1)[:, -k:]
        mask = np.zeros_like(m, dtype=int)
        rows = np.arange(m.shape[0])[:, None]
        mask[rows, topk_idx_partition] = 1
        return mask

def from_correlation_values_to_score(m):
    mask = np.abs(copy.deepcopy(m))  # this is what we use to remove the rows
    result = np.zeros((m.shape))
    score = 0
    for i in range(mask.shape[0]):  # its a square
        largest_x, largest_y = np.unravel_index(np.argmax(mask, axis=None), mask.shape)  # this is straight from numpy docs
        mask[largest_x,:] = 0
        mask[:, largest_y] = 0
        result[largest_x, largest_y] = m[largest_x, largest_y]
        score += abs(m[largest_x, largest_y])
    return result, score/float(m.shape[0])


def are_the_peaks_ok(peaks_a, peaks_b, m, p_values=None):
    #both a binary matrix and score matrix can be handled the same way.
    found_correspondence = {}
    signf = {}
    num_peaks_a = float(len(peaks_a))
    num_peaks_b = float(len(peaks_b))
    if num_peaks_a == 0 or num_peaks_b == 0: # nothing to compare against, will result in zero division
        return 0, [0], {}
    # see which peaks correlate; loop over all combinations
    # matrix (greedy) construction guarantees that the matrix is a permutation
    # argmax construction does not, still, this adequately measures the peaks most of the time
    # outside complicated edge cases.
    
    for a,b in product(peaks_a, peaks_b):
        #print(f"in peaks, {a,b}: {m[a,b]}")
        if np.abs(m[a,b]) > 0:  
            found_correspondence[a] = b
            if p_values is not None:
                signf[f"{a}-{b}"] = p_values[a,b]
            else:
                signf[f"{a}-{b}"] = 1   # not significant because we do not know
    # if these sets are the same, then everything is good
    #print(f"found correpondence {found_correspondence}")

    # For each return a fraction of correct answers, their correlations, and significance
    if set(peaks_a) == set(found_correspondence.keys()) and set(peaks_b)== set(found_correspondence.values()):
        return 1, [m[a,b] for a,b in found_correspondence.items()], signf
    else:
        # partial perfect match
        if set(peaks_a) == set(found_correspondence.keys()):
            return num_peaks_a/num_peaks_b, [m[a,b] for a,b in found_correspondence.items()], signf
        elif set(peaks_b) == set(found_correspondence.values()):
            return num_peaks_b/num_peaks_b, [m[a,b] for a,b in found_correspondence.items()], signf
        elif len(found_correspondence) > 0:
            # imperfect partial match
            return len(found_correspondence)/max(num_peaks_a, num_peaks_b), [m[a,b] for a,b in found_correspondence.items()], signf
        else:  # no match at all
            return 0, [m[a,b] for a,b in found_correspondence.items()], signf
    


  

def matrix_permutation_correlation(two_matrices, peak_dims, method="greedy", plots=False, prints=False):
    assert len(two_matrices) == 2, "Give two matrices for matrix_permutation_correlation"
    assert len(peak_dims) == 2, "number of dimensions does not match matrices in matrix_permutation_correlation"
    a, b = two_matrices[0], two_matrices[1]
    dim_a, dim_b = peak_dims[0], peak_dims[1]
    if prints:
        print(f"peaks are {dim_a}, {dim_b}")
    results = np.zeros((a.shape[0], a.shape[0]))
    p_values = np.zeros((a.shape[0], a.shape[0]))
    for i, rowa in enumerate(a):
        for j, rowb in enumerate(b):
            r = pearsonr(rowa, rowb)  # spearmanr also option, but pearson used in related work
            results[i,j] = r.statistic
            p_values[i,j] = r.pvalue
    # show the cross correlation matrix:
    if plots:
        plot_correlation_heatmap(results, [dim_a, dim_b])
    # get from the heatmap to a score
    continuous_mask, score = from_correlation_values_to_score(results)
    #if plots:
    #    plot_correlation_heatmap(continuous_mask,[dim_a, dim_b])
    perm = check_if_a_permutation_matrix(continuous_mask)
    assert perm, "⚠️ NOT A PERMUTATION ⚠️"
    if prints:
        print(f"Correlation score for these two models is {score} ✅, with permutation = {perm}")
    peak_ok, peak_coeffs, peak_signf = are_the_peaks_ok(dim_a, dim_b, continuous_mask, p_values=p_values)
    if prints:
        if peak_ok == 1:
            print(f"the peaks correlate for 2 models: {peak_coeffs} with significance {peak_signf} ✅")
        elif peak_ok > 0:
            print(f"the peaks partially correlate for 2 models: {peak_coeffs} with significance {peak_signf} 🤔 ")
        else:
            print(f"⚠️ peaks do not correlate for these two models, score is {peak_coeffs}")
    return score, peak_ok, peak_coeffs, peak_signf

In [ ]:
method = "ICA"
dim = 32
model_to_analyse = ["bge-m3"]
lang_to_analyse = ["en-de", "en-bg", "en-uk", "en-it", "en-fr", "en-vi", "en-el", "en-ja", "en-hi", "en-fi"]
plot=False
prints=False
for lang_path in glob(f"stability/{method}_dim_{dim}/tatoeba/*/"):
    lang = lang_path.split("/")[-2]
    print(lang)
    if lang_to_analyse != "all":
        if lang not in lang_to_analyse:
            continue
    files = glob(lang_path+"/*")
    unique_models = np.unique([basename(f).split("_")[0] for f in files])
    print(unique_models)
    for m in unique_models:
        if m not in model_to_analyse and model_to_analyse != "all":
            continue
        print(f"In {m}")
        files_to_read = glob(f"{lang_path}/{m}_results_*[!f].pkl") # !f removes unneeded "diff" files
        if len(files_to_read) < 3:
            print("Too little files.")
            continue
        perms = np.zeros((len(files_to_read), len(files_to_read)))
        peaks = np.zeros((len(files_to_read), len(files_to_read)))
        peaks_signf = np.zeros((len(files_to_read), len(files_to_read)))
        stats = {}
        for i1, i2 in combinations(range(len(files_to_read)), 2):  # do this to get model comparison numbers
            #print(f"COMPARING MODEL ITERATIONS {i1} and {i2}")
            f1 = files_to_read[i1]
            f2 = files_to_read[i2]
            data1, model1 = load_data_and_matrix(f1)
            data2, model2 = load_data_and_matrix(f2)
            if plot:
                plot_the_bar_graph(data1, method,f"tatoeba:{lang}", f"{m}-{i1}")
            dim1, st1 = calculate_statistics(data1, method)
            stats[f1] = st1
            if plot:
                plot_the_bar_graph(data2, method,f"tatoeba:{lang}", f"{m}-{i2}")
            dim2, st2 = calculate_statistics(data2, method)
            stats[f2] = st2
            #perm_score, peak_score, peak_coeffs, peak_signf = matrix_permutation_correlation([model1, model2], [dim1, dim2], plots=plot, prints=prints)
            #perms[i1, i2] = perm_score
            #peaks[i1, i2] = peak_score * np.mean(np.abs(peak_coeffs))
            #print(peak_signf)
            #peaks_signf[i1, i2] = [i for i in peak_signf.values()][0]
        #print(stats)
        summary, stat_check = check_consistency_of_statistics([v for _,v in stats.items()], prints=prints)
        print("STATS CHECK", stat_check)
        if all(stat_check.values()):
            print(f"Consistent metrics for {m}-{lang}")
        else:
            print(f"Inconsistent with \n{stat_check}")
        continue
        lang_mapped = "eng-"+lang_map2[lang.split("-")[-1]]
        plot_correlation_heatmap(perms, [[],[]], title = f"Cross-model correlation, {m}-Tatoeba:{lang}")
        plot_correlation_heatmap(peaks, [[],[]], title = f"Cross-peak correlation, {m}-Tatoeba:{lang}", p_values = peaks_signf)
        #print(peaks_signf)
        print("")

# "Unit tests"

In [ ]:
# permutation detection

a = np.eye(5)
random_row_permutation = np.random.permutation(len(a))
b = a[random_row_permutation,:]
c = a[random_row_permutation,:]
d = a[random_row_permutation,:]
# add one 1 to random non-1 place in c
zero_indices = np.where(b == 0)
r = np.random.choice(range(len(zero_indices[0])))
c[zero_indices[0][r], zero_indices[1][r]] = 1
# remove one 1 in random place in c
one_indices = np.where(b == 1.)
r = np.random.choice(range(len(one_indices[0])))
d[one_indices[0][r], one_indices[1][r]] = 0
# a and b should be, c and d not
assert check_if_a_permutation_matrix(a)
assert check_if_a_permutation_matrix(b)
assert check_if_a_permutation_matrix(c) == False
assert check_if_a_permutation_matrix(d) == False

In [ ]:
# from_correlation_values_to_score(m)

m = np.array([[0.35, -0.30, 0.67, 0.79, 0.03],
            [0.001, 0.50, 0.99, 0.151, -0.46],
            [0.92, 0.321, 0.60, -0.309, 0.54],
            [-0.148, -0.88, 0.623, 0.31, 0.207],
            [0.606, 0.27, 0.29, 0.15, -0.899]])
plot_correlation_heatmap(m, [[1],[2]])
r, score = from_correlation_values_to_score(m)
plot_correlation_heatmap(r, [[1],[2]])
print(f"over all alignment score: {score}")
peak_score = are_the_peaks_ok([1],[2], r)
print(f"Peak score {peak_score}")